# The Fix That Disappears: reproducing the paper

This notebook rebuilds the tables and figures of the paper from the released transcripts of GenLineage-Bench. It needs no GPU and no model. Every number is recomputed from the raw run logs by `scripts/paper_stats.py`, the same script the paper's claims checker uses.

The last section is a small live demo on CPU. It takes one benchmark task, applies a fix to the generated file and a fix to the source, and shows that only the second one survives `make gen`. It then runs the LineageGuard write gate on the first fix.

Code and data: https://github.com/Hotragn/genlineage-bench (Apache 2.0). The paper PDF is in the repository under `paper/`.

## 1. Get the repository

On Kaggle this cell clones the public repository at the release tag, so the notebook does not depend on any attached data. Locally, run the notebook next to `genlineage-bench/` or set `GL_REPO`, and nothing is downloaded.

In [ ]:
import glob, json, os, subprocess, sys, tarfile

GIT_URL = "https://github.com/Hotragn/genlineage-bench"
GIT_TAG = "v1.0-kaggle"

def find_repo():
    cands = [os.environ.get("GL_REPO", ""), "genlineage-bench", "../genlineage-bench", "/tmp/genlineage-bench"]
    cands += glob.glob("/kaggle/input/*/genlineage-bench") + glob.glob("/kaggle/input/*")
    for c in cands:
        if c and os.path.exists(os.path.join(c, "scripts", "paper_stats.py")):
            return os.path.abspath(c)
    # a release tarball next to the notebook also works
    for t in glob.glob("/kaggle/input/*/genlineage-bench.tar.gz") + glob.glob("genlineage-bench.tar.gz") + glob.glob("../package/genlineage-bench.tar.gz"):
        with tarfile.open(t) as tf:
            tf.extractall("/tmp/gl_unpack")
        return "/tmp/gl_unpack/genlineage-bench"
    # otherwise clone the public repository at the tag the paper was checked against
    # clone outside /kaggle/working so the repository is not saved as notebook output
    dest = "/tmp/genlineage-bench" if os.path.isdir("/kaggle/working") else os.path.abspath("genlineage-bench")
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", GIT_TAG, GIT_URL, dest], check=True)
    print(subprocess.run(["git", "-C", dest, "log", "-1", "--format=commit %H (%cI)"], capture_output=True, text=True).stdout.strip())
    return dest

REPO = find_repo()
if not os.access(os.path.join(REPO, "results", "summary"), os.W_OK):  # Kaggle inputs are read-only
    import shutil
    shutil.copytree(REPO, "/tmp/genlineage-bench", dirs_exist_ok=True); REPO = "/tmp/genlineage-bench"
sys.path.insert(0, os.path.join(REPO, "scripts"))
print("repository:", REPO)

## 2. Recompute every statistic from the raw transcripts

This reads all released runs under `results/` and writes `results/summary/paper_stats.json`. It takes about a minute on CPU, most of it for the task-clustered bootstrap.

In [ ]:
def run(*args):
    r = subprocess.run([sys.executable, *args], cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()

run("scripts/paper_stats.py")
P = json.load(open(os.path.join(REPO, "results", "summary", "paper_stats.json")))
print(len(P), "values recomputed")
print(f"Natural agents passed the tests in {P['pool.natural.imm']} of {P['pool.natural.n']} runs on regen and mixed tasks.")
print(f"{P['pool.natural.lost']} of those {P['pool.natural.imm']} fixes disappeared after make gen (DFR {P['pool.natural.DFR']}).")
print(f"Preregistered DRR: natural {P['pool.natural.DRR']}, hint {P['pool.hint.DRR']}, HeaderGuard {P['pool.headerguard.DRR']}, "
      f"R+G {P['pool.gate.DRR']}, R+G+V {P['pool.full.DRR']} (oracle-in-the-loop upper bound).")

## 3. Table 1: preregistered DRR per model and condition

Regen and mixed tasks, seeds 1 and 2. Percent of runs, with the task-clustered 95% interval for DRR. Strict DRR is kept out of this table on purpose; it is in Figure 3.

In [ ]:
import pandas as pd
MODELS = [("k_e2b", "E2B"), ("k_e4b", "E4B"), ("k_12b", "12B")]
CONDS = [("natural", "natural"), ("hint", "hint"), ("headerguard", "HeaderGuard"), ("gate", "R+G"), ("full", "R+G+V")]
t1 = pd.DataFrame([{"Model": mn, "Condition": cn, "ISR": P[f"{m}.{c}.ISR"],
                    "DRR [95% CI]": f"{P[f'{m}.{c}.DRR']} {P[f'{m}.{c}.DRR_ci']}", "DFR": P[f"{m}.{c}.DFR"]}
                   for m, mn in MODELS for c, cn in CONDS])
t1

## 4. Table 2: per-model paired contrasts

DRR change in points, then discordant pairs (treatment-only:baseline-only) and the unadjusted task-level permutation p.

In [ ]:
PAIRS = [("natural_vs_full", "natural vs R+G+V"), ("hint_vs_full", "hint vs R+G+V"),
         ("headerguard_vs_full", "HeaderGuard vs R+G+V"), ("headerguard_vs_gate", "HeaderGuard vs R+G")]
t2 = pd.DataFrame([{"Model": mn, **{lab: f"{P[f'{m}.{k}.diff']} ({P[f'{m}.{k}.colon']}; {P[f'{m}.{k}.p']})" for k, lab in PAIRS}}
                   for m, mn in MODELS])
print("Pooled, task-level Holm p:", P["tl.natural_vs_full.holm"], P["tl.hint_vs_full.holm"], P["tl.headerguard_vs_full.holm"])
print("12B natural vs R+G+V, per-model Holm p:", P["k_12b.natural_vs_full.holm"], "(not significant, so no 12B effect is claimed)")
t2

## 5. Figures

The two diagrams are drawn by `scripts/fig_diagrams.py`. Figures 3 and 4 are drawn from the released data.

In [ ]:
from IPython.display import Image, display
OUT = os.path.abspath("figures"); os.makedirs(OUT, exist_ok=True)
for s in ("fig_diagrams.py", "paper_figures.py", "fig_aidev.py"):
    run(os.path.join("scripts", s), OUT)
for f in ("fig1_disappearing_fix.png", "fig2_overview.png", "fig3_immediate_vs_durable.png", "fig4_aidev.png"):
    print(f); display(Image(os.path.join(OUT, f), width=900))

## 6. Claims check

This renders the paper template with the recomputed values and requires the result to match `paper/writeup.md` byte for byte. It also audits the word count and cross-checks the cell counts against two independently written analyses.

In [ ]:
run("scripts/check_claims.py", "--no-recompute")

## 7. Live demo on one task (CPU, optional)

Task `oa1_property_typo`: the OpenAPI spec misspells `email` as `emial`, and `client/models.py` is generated from it. We score two patches with the durable scorer: one edits the generated file, the other edits the spec. Then we ask the LineageGuard write gate what it would do with the first patch.

This needs the pinned generators (`pip install -r genlineage/requirements.txt`). Set `RUN_LIVE_DEMO = False` to skip it.

In [ ]:
RUN_LIVE_DEMO = True
import importlib.util, shutil, tempfile
have_gen = importlib.util.find_spec("datamodel_code_generator") is not None
if RUN_LIVE_DEMO and not have_gen:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO, "genlineage", "requirements.txt")])
    have_gen = importlib.util.find_spec("datamodel_code_generator") is not None
if RUN_LIVE_DEMO and have_gen:
    sys.path.insert(0, REPO)
    from genlineage.score import apply_patch, score
    from genlineage.env import sh
    from genlineage.lineage import infer, edit_survives, pointer
    task = os.path.join(REPO, "bench", "tasks", "oa1_property_typo")
    meta = json.load(open(os.path.join(task, "task.json"))); orig = os.path.join(task, "repo")
    work = tempfile.mkdtemp(prefix="gl_demo_")
    for name, patch, regen in (("edit the generated file", meta["gen_only_fix"], False), ("edit the source spec", meta["ref_fix"], True)):
        w = os.path.join(work, name.replace(" ", "_")); shutil.copytree(orig, w)
        apply_patch(w, patch)
        if regen: sh(meta["gen_cmd"], w)
        s = score(meta, w, orig)
        print(f"{name:24s} tests pass now: {s['immediate_pass']!s:5s}  after make gen: {s['durable_pass']}")
    lin = infer(orig, commands=[meta["gen_cmd"]])
    rel = "client/models.py"; old = open(os.path.join(orig, rel)).read(); new = old.replace("emial", "email")
    ok, detail = edit_survives(orig, meta["gen_cmd"], rel, new, old)
    print("\nwrite gate on the generated-file edit:", "accept" if ok else "reject")
    print(detail[:300])
    print(pointer(lin, rel, old, new).strip())
    shutil.rmtree(work, ignore_errors=True)
else:
    print("live demo skipped")